# 🏦 JPMC Consumer Banking - Hands-on Lab
## Notebook 10: Connect Snowsight Workspace to GitHub Repository

---

### What This Notebook Does
Sets up the prerequisites so participants can create a **Git-synced Workspace** in Snowsight, connected to the HOL GitHub repository.

**Repository:** `https://github.com/sfc-gh-vthota/JPMC-HOL-DATAENGINEERING.git`

### What is a Git Workspace?
A Git workspace in Snowsight lets you:
- Work directly on files from a Git branch inside Snowsight
- Pull latest changes from the remote repo
- Create branches, edit files, and push commits back to GitHub
- View diffs and resolve merge conflicts - all within Snowsight

### Prerequisites (done in this notebook)
1. An **API Integration** that allows access to GitHub
2. (Optional) A **Secret** with a GitHub PAT for private repos or write access

### Participant Steps (done in the Snowsight UI)
1. Navigate to **Projects → Workspaces**
2. Select **From Git repository**
3. Paste the repo URL
4. Select the API integration
5. Choose authentication method

In [ ]:
-- =============================================================================
-- SETUP CONTEXT
-- =============================================================================
USE ROLE ACCOUNTADMIN;

## Step 1: Create API Integration for GitHub

The API Integration allows Snowflake workspaces to communicate with GitHub. This must be created by ACCOUNTADMIN.

In [ ]:
-- =============================================================================
-- STEP 1: Create API Integration for Git HTTPS
-- This allows Snowsight Workspaces to connect to GitHub repositories
-- =============================================================================

CREATE OR REPLACE API INTEGRATION HOL_GIT_API_INTEGRATION
    API_PROVIDER = GIT_HTTPS_API
    API_ALLOWED_PREFIXES = ('https://github.com/sfc-gh-vthota/')
    ALLOWED_AUTHENTICATION_SECRETS = ALL
    ENABLED = TRUE
    COMMENT = 'API integration for JPMC HOL GitHub repository - used by Git Workspaces';

-- Grant usage to SYSADMIN so participants can use it
GRANT USAGE ON INTEGRATION HOL_GIT_API_INTEGRATION TO ROLE SYSADMIN;

## Step 2: (Optional) Create a Secret for GitHub PAT

Since this is a **public repository**, you can skip this step and select "Public repository" as the authentication method in the UI.

However, if you want write access (to commit/push changes back), create a secret with a GitHub Personal Access Token:

In [ ]:
-- =============================================================================
-- STEP 2 (OPTIONAL): Create a Secret for write access via PAT
-- Skip this if using "Public repository" authentication (read-only)
-- =============================================================================

-- To generate a PAT: GitHub → Settings → Developer settings → Personal access tokens
-- Required scopes: repo (full control) for push access

CREATE DATABASE IF NOT EXISTS JPMC_CONSUMER_BANKING_HOL;
CREATE SCHEMA IF NOT EXISTS JPMC_CONSUMER_BANKING_HOL.HOL_LAB;

CREATE OR REPLACE SECRET JPMC_CONSUMER_BANKING_HOL.HOL_LAB.HOL_GIT_SECRET
    TYPE = PASSWORD
    USERNAME = 'sfc-gh-vthota'
    PASSWORD = '<<REPLACE_WITH_YOUR_GITHUB_PAT>>'
    COMMENT = 'GitHub PAT for push access to JPMC HOL repo';

-- Grant usage so participants can reference this secret
GRANT USAGE ON SECRET JPMC_CONSUMER_BANKING_HOL.HOL_LAB.HOL_GIT_SECRET TO ROLE SYSADMIN;

## Step 3: Create the Git Workspace in Snowsight (UI Steps)

Now that the API Integration is ready, follow these steps in the **Snowsight UI**:

### Instructions for Participants

1. Sign in to **Snowsight**
2. Navigate to **Projects → Workspaces**
3. Click the **+ (New)** dropdown → Select **From Git repository**
4. Paste the repository URL:
   ```
   https://github.com/sfc-gh-vthota/JPMC-HOL-DATAENGINEERING.git
   ```
5. (Optional) Rename the workspace (e.g., `JPMC-HOL-DataEngineering`)
6. In the **API Integration** dropdown, select: `HOL_GIT_API_INTEGRATION`
7. For **Authentication**, choose one of:
   - **Public repository** → Read-only access (recommended for HOL participants)
   - **Personal access token** → Select the secret `HOL_GIT_SECRET` for read+write access
8. Click **Create**

### After Creation
- The workspace opens showing all files from the `main` branch
- You can open any `.ipynb` or `.sql` file directly in Snowsight
- Use the **Changes** tab to see the current branch and pull updates

## Step 4: Working with the Git Workspace

Once connected, here's how to use the Git workspace:

### Pull Latest Changes (when instructor updates the repo)
1. Open your workspace
2. Click the **Changes** tab
3. Click **Pull** to fetch and merge latest from remote

### Create a Branch (to experiment without affecting main)
1. In the **Changes** tab, click the branch dropdown
2. Select **+ New**
3. Name your branch (e.g., `participant-01-experiments`)
4. Click **Create**

### Commit and Push (if you have write access)
1. Make edits to files in the workspace
2. Go to **Changes** tab - modified files show with an **M** indicator
3. Write a commit message
4. Click **Push**

### Switch Branches
1. In the **Changes** tab, click the branch dropdown
2. Select the branch you want to switch to

### Fetch Remote Branches
1. In the **Changes** tab, click the down arrow next to **Pull**
2. Select **Fetch All** to see branches created outside Snowsight

In [ ]:
-- =============================================================================
-- VERIFICATION: Confirm the API Integration is ready
-- =============================================================================

DESCRIBE INTEGRATION HOL_GIT_API_INTEGRATION;

-- List existing secrets (if PAT was created)
SHOW SECRETS IN SCHEMA JPMC_CONSUMER_BANKING_HOL.HOL_LAB;

## Summary

| What | How |
|------|-----|
| **API Integration** | `HOL_GIT_API_INTEGRATION` (created via SQL above) |
| **Repo URL** | `https://github.com/sfc-gh-vthota/JPMC-HOL-DATAENGINEERING.git` |
| **Auth (read-only)** | Select "Public repository" in the UI |
| **Auth (read+write)** | Use PAT secret `HOL_GIT_SECRET` |
| **Create workspace** | Snowsight → Projects → Workspaces → From Git repository |

### Key Points
- The SQL setup (API Integration) must be done **once per account** by ACCOUNTADMIN
- Each participant creates their own Git workspace from the UI
- Public repo = read-only (participants can pull but not push)
- PAT auth = read+write (participants can commit and push changes)
- All HOL notebooks are immediately available in the workspace after connection